# The Street Network Graph


**Network analysis** treats features as elements of a network joined by links, and measures everything **along that network** — along streets, along transit routes — rather than in a straight line. That one change makes models of movement and accessibility in a city far more realistic.

The formal framework behind it is **graph theory**, the branch of mathematics that deals with structures made of nodes and edges.

A **graph** is a mathematical model of a network: a set of nodes (vertices) and edges (connections) between them.

In the context of a street network:

- **nodes** — intersections, junctions, or dead ends,
- **edges** — road segments connecting the nodes.

![Graph](images/network.png)

_Own figure._

In this section, we will retrieve a street network graph from OpenStreetMap (OSM) data and explore its key properties.

## 0. Importing Libraries


In [ ]:
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt

# cache OSM responses on disk, so repeating a query does not hit the server again
ox.settings.cache_folder = "../../cache"


- [**NetworkX**](https://networkx.org/) (`networkx`) — a Python library for creating, analysing, and visualising graphs and networks. It provides tools for working with nodes and edges, computing centrality metrics, finding shortest paths, and exploring network structure. Widely used for network analysis in transportation, social networks, and spatial analytics.


## 1. Street Network Graph from OSMnx

The `osmnx` library provides convenient tools for retrieving and analysing street networks from OpenStreetMap data.

In `osmnx`, the network is represented as a directed multigraph (`MultiDiGraph`):

- edges have direction (e.g. one-way streets are taken into account),
- multiple edges can exist between the same pair of nodes,
- edges carry attributes such as length, road type, travel restrictions, and other properties.

This representation enables a range of network analysis methods: routing, accessibility assessment, network structure analysis, and isochrone generation.


### 1.1. Retrieving the Street Network Graph

Let's retrieve the street network graph for a chosen area from OpenStreetMap.

We use the `ox.graph_from_place()` function, which downloads OSM data for a given place name and converts it into a graph automatically.

The key parameters are:

- the place name (`area_name`),
- the network type (`network_type`), which determines which roads are included in the graph (e.g. drivable, walkable, or cyclable).


_In the previous sections, we worked with the Innere Stadt, the first district of Vienna. Here we move one district out, to Landstraße: the Innere Stadt is largely pedestrianised, and a drivable network full of gaps and one-way loops makes the metrics below harder to read than they need to be._


In [ ]:
# Define the area
area_name = "Landstraße, Vienna, Austria"

# Retrieve the street network graph from OpenStreetMap
# network_type='drive' — drivable network (roads accessible by car)
graph = ox.graph_from_place(area_name, network_type="drive")

# Visualise the graph
ox.plot_graph(graph)

## 2. Basic Network Properties

Once the street network graph has been retrieved, we can calculate some basic properties. These give an initial picture of the network's structure and serve as a starting point for further analysis.

Key properties include:

- the number of nodes (intersections and street endpoints),
- the number of edges (road segments),
- the degree distribution of nodes (intersection structure).


### 2.1. Node and Edge Count

The simplest graph property is the number of nodes and edges. These figures give a sense of the overall size of the network.


In [ ]:
# Count nodes and edges
num_nodes = len(graph.nodes)
num_edges = len(graph.edges)

print(f"Number of nodes: {num_nodes}")
print(f"Number of edges: {num_edges}")

### 2.2. Node Degree Distribution

The **degree** of a node is the number of edges connected to it.
In a street network, this reflects the structure of intersections.

For example:

- a node with degree 1 typically corresponds to a dead end,
- a node with degree 3 or 4 corresponds to a standard intersection,
- nodes with higher degrees may correspond to major junctions.

Analysing the degree distribution helps characterise how well-connected the network is and how its intersections are structured.


In [ ]:
# Get node degrees
node_degrees = dict(graph.degree())

# Extract degree values
degree_values = list(node_degrees.values())

# Plot the histogram
plt.figure()
plt.hist(degree_values, bins=range(min(degree_values), max(degree_values)+2))

plt.xlabel("Node degree")
plt.ylabel("Number of nodes")
plt.title("Node degree distribution")

plt.show()

The degree distribution reveals the character of the street network. A large number of degree-1 nodes indicates the presence of dead ends, while a predominance of degrees 3–4 or higher is typical of a regular urban grid.


### 2.3. Connectivity

Network connectivity describes how well the elements of a network are linked to one another.

At a basic level, connectivity can be assessed using:

- the number of connected components,
- the size of the largest connected component.

A **connected component** is a subgraph in which every node can be reached from every other node.
If there is more than one component, the network contains isolated, disconnected parts.


In [ ]:
# Convert to undirected graph for connectivity analysis
graph_undirected = graph.to_undirected()

# Find connected components
components = list(nx.connected_components(graph_undirected))

print(f"Number of connected components: {len(components)}")
print(f"Size of the largest component: {len(max(components, key=len))}")

## 3. Reprojecting the Graph


When working with street networks, keep an eye on the coordinate reference system the data is in.

Graphs retrieved with `osmnx` are in the **geographic coordinate system** WGS 84 (EPSG:4326) by default, with coordinates expressed in degrees.

This is convenient for storage and visualisation, but can be limiting for certain types of analysis.

The graph already includes a `length` attribute on each edge, storing the road length in metres — so many algorithms (such as shortest path) work correctly even without reprojection. However, for geometry-based operations such as buffering or distance measurement, reprojection is recommended.

The `osmnx` library can automatically reproject the graph into an appropriate projected CRS (typically UTM) using `ox.project_graph()`.


In [ ]:
graph_utm = ox.project_graph(graph)

The CRS after reprojection:


In [ ]:
graph_utm.graph["crs"]

The projected copy is stored in `graph_utm`. In the next sections we keep working with the original graph: its edges already carry `length` in metres, which is what the routing and centrality algorithms use. The projected version is what you would reach for when a task needs the geometry itself — buffers, areas, or distances measured off the network.

## 4. Converting the Graph to GeoDataFrames

For spatial analysis and visualisation, it is sometimes useful to convert the graph into GeoDataFrames. In this form, nodes and edges are represented as spatial features (points and lines) that can be processed using standard geoprocessing operations.

This is done with `ox.graph_to_gdfs()`, which returns two GeoDataFrames:

- `nodes` — a point layer containing the nodes,
- `edges` — a line layer containing the edges.

Let's convert the graph and look at the size of both layers, then plot the edges.

In [ ]:
# Convert the graph to GeoDataFrames
nodes, edges = ox.graph_to_gdfs(graph)

print(f"Nodes layer: {nodes.shape}")
print(f"Edges layer: {edges.shape}")

# Plot the edges
edges.plot(figsize=(10, 10), color="blue")

## Summary


In this section, we introduced the street network graph and explored how to work with it.

We learned:

- how to retrieve a street network graph from OpenStreetMap;
- how a graph is structured (nodes and edges) and what attributes it contains;
- how to calculate basic network properties: node and edge counts, degree distribution, and connectivity;
- how to convert a graph to GeoDataFrames for spatial analysis.

In this example, we analysed the street network of a specific district. The same approach applies to any area you want to study.

In the next section, we will move on to specific network analysis tasks.
